# Ablation Notebook 2B — Unlearning μ-Source Ablation · Group B (3/7 Split)

**Experiment:** CMF global-mean μ source ablation under a 30% forget ratio (3/7 split).  

**Prerequisite:** Run **Ablation Notebook 1** first and attach its output dataset.  
Set `CKPT_DATASET_DIR` in Section A to the dataset mount path.  
Run **Ablation Notebook 2A** for Group A methods.

**Group B methods (this notebook):**

| Key | Paper name | mean_source runs |
|-----|-----------|------------------|
| `random_label_CMF_RemoveFC` | Random Label w. CMF | train  +  retain |
| `salun_CMF_RemoveFC` | SalUn w. CMF | train  +  retain |

Each method is run **twice** — once with `mean_source='train'` (legacy, μ includes forget  
samples) and once with `mean_source='retain'` (ablation fix, μ from retain set only).  

| Stage | Description |
|-------|-------------|
| **A** | Environment setup, repo clone, load ablation_config.json |
| **B** | Group B method selection |
| **C** | Dataset & data-loaders (3/7 split) |
| **D** | Run Group B ablation (2 methods × 2 mean_source modes) |
| **E** | Results table + delta comparison |
| **F** | μ drift plot |
| **G** | Bar chart |
| **H** | t-SNE |
| **I** | Combined summary (merge Group A CSV if present) |

> **Recommended:** GPU T4/P100.  
> Group B (2 methods × 2 modes = 4 runs) ≈ 1–2 h on T4 in full mode.

## A. Environment Setup & Load Config

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print('STDERR:', r.stderr[-2000:])
    return r.returncode

sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, copy, random, argparse, collections, math, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ======================================================================
#  SET THIS to the Kaggle dataset mount path from Ablation Notebook 1.
#  Typical: /kaggle/input/<your-ablation-dataset-slug>/
# ======================================================================
CKPT_DATASET_DIR = '/kaggle/input/cmf-ablation-37-checkpoints'  # <- EDIT THIS

# Optional: attach the Group A output CSV to merge into the final table.
# Set to None if Group A has not been run yet.
GRPA_RESULTS_CSV = None  # e.g. '/kaggle/input/cmf-ablation-grpA/ablation_results_grpA_cifar10_resnet18.csv'

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/ablation_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/ablation_37/ablation_config.json',
    f'{CKPT_DATASET_DIR}/ablation_37/ablation_config.json',
]

config_path = None
CKPT_ROOT   = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p
        CKPT_ROOT   = os.path.dirname(_p)
        break

assert config_path is not None, (
    'ablation_config.json not found. Checked:\n' +
    '\n'.join(f'  - {p}' for p in _CONFIG_CANDIDATES) + '\n'
    'Make sure Ablation Notebook 1 finished and the correct dataset is attached.')

with open(config_path) as f:
    CFG = json.load(f)

TEST_MODE         = CFG['TEST_MODE']
TEST_FRACTION     = CFG['TEST_FRACTION']
_MODE_TAG         = CFG['_MODE_TAG']
DATASET           = CFG['DATASET']
ARCH              = CFG['ARCH']
IS_VIT            = CFG['IS_VIT']
SEED              = CFG['SEED']
NUM_CLASSES       = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES = CFG['CLASS_LABEL_NAMES']
FORGET_CLASSES    = list(CFG['FORGET_CLASSES'])
_FORGET_STR       = CFG['FORGET_STR']
NUM_FORGET        = int(CFG['NUM_FORGET'])
NUM_RETAIN        = int(CFG['NUM_RETAIN'])
PRETRAIN_LR       = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS   = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS       = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']
RETRAIN_EPOCHS    = CFG['RETRAIN_EPOCHS']
_TOTAL     = {DATASET: CFG['TOTAL']}
_PER_CLASS = {DATASET: CFG['PER_CLASS']}

_old_root = CFG['CKPT_ROOT']
def _repath(p):
    return p.replace(_old_root, CKPT_ROOT)

CKPT_PRETRAIN = _repath(CFG['CKPT_PRETRAIN'])
CKPT_CMF_FT   = _repath(CFG['CKPT_CMF_FT'])

for p, name in [(CKPT_PRETRAIN, 'pre_train'), (CKPT_CMF_FT, 'CMF_FT')]:
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}] {name}: {p}')

DATA_PATH = '/kaggle/working/data'
WORK_ROOT = '/kaggle/working'
os.makedirs(DATA_PATH, exist_ok=True)

print(f'\nLoaded config: {config_path}')
print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Forget classes : {FORGET_CLASSES}  (3/7 split)')
print(f'Forget={NUM_FORGET}  Retain={NUM_RETAIN}')

## B. Method Selection — Group B

Group B: **Random-label** and **SalUn** — both run with `mean_source='train'` AND `mean_source='retain'`.  
Group A (Ablation Notebook 2A): SCRUB and NegGrad+.

In [ ]:
# Group B methods
RUN_METHODS = [
    'random_label_CMF_RemoveFC', # Random Label w. CMF  (priority 3)
    'salun_CMF_RemoveFC',        # SalUn w. CMF  (priority 4)
]

# Both mu-source modes for every method
MEAN_SOURCES = ['train', 'retain']

UNLEARN_BS = 8 if TEST_MODE else 128

_LR = {
    'random_label_CMF_RemoveFC': {'cifar10': {'s': 1e-4, 'm': 2e-3}},
    'salun_CMF_RemoveFC':        {'cifar10': {'s': 2e-4, 'm': 2e-3}},
}
_EPOCHS = {
    'random_label_CMF_RemoveFC': 4,
    'salun_CMF_RemoveFC':        4,
}
if TEST_MODE:
    _EPOCHS = {k: 1 for k in _EPOCHS}

def get_lr(method):
    # 3 forget classes -> multi mode ('m')
    return _LR.get(method, {}).get(DATASET, {}).get('m', 1e-3)

def get_epochs(method):
    return _EPOCHS.get(method, 1 if TEST_MODE else 3)

print(f'Group B methods ({len(RUN_METHODS)}): {RUN_METHODS}')
print(f'mean_sources: {MEAN_SOURCES}')
print(f'Total runs: {len(RUN_METHODS)} x {len(MEAN_SOURCES)} = {len(RUN_METHODS)*len(MEAN_SOURCES)}')

## C. Dataset & Data-Loaders (3/7 Split)

In [ ]:
from utils import get_dataset, get_model, get_retain_forget_partition, test, load_encoder_ckpt_safely
from unlearn import unlear_func
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=SEED, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=NUM_RETAIN, num_forget_samples=NUM_FORGET,
        grad_norm_clip=1.0, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=True, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=True, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT, nc_pool_mode='avg',
        project_name='kaggle_ablation_37', group_name='mu_source_grpB',
    )
    d.update(ov)
    return argparse.Namespace(**d)

args_base = make_args(unlearn_class=list(FORGET_CLASSES))
dataset_train, dataset_test = get_dataset(args_base)

if TEST_MODE:
    def _stratified_subset(ds, fraction, seed=SEED):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]; rng.shuffle(cls_pool)
            kept.extend(cls_pool[:max(1, math.ceil(len(cls_pool) * fraction))])
        sub = torch.utils.data.Subset(ds, kept)
        base_tgt = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_tgt[i] for i in kept]
        return sub
    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True,
                 shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, max(1, len(dataset_test))), num_workers=2,
                 pin_memory=True, shuffle=False)

train_loader = torch.utils.data.DataLoader(dataset_train, **LOADER_KW)
test_loader  = torch.utils.data.DataLoader(dataset_test,  **TEST_KW)

args_part = make_args(unlearn_class=list(FORGET_CLASSES))
retain_ds, forget_ds = get_retain_forget_partition(args_part, dataset_train, FORGET_CLASSES)
retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
forget_loader = torch.utils.data.DataLoader(forget_ds, **LOADER_KW)
_, test_forget_ds = get_retain_forget_partition(args_part, dataset_test, FORGET_CLASSES)
test_forget_loader = torch.utils.data.DataLoader(test_forget_ds, **TEST_KW)

retain_full_loader = torch.utils.data.DataLoader(
    retain_ds,
    batch_size=min(256, max(1, len(retain_ds))),
    shuffle=False, num_workers=2, pin_memory=True,
)

print(f'Train={len(dataset_train)}  Test={len(dataset_test)}')
print(f'Forget={len(forget_ds)}  Retain={len(retain_ds)}  TestForget={len(test_forget_ds)}')

args_pt = make_args(unlearn_method='pre_train', remove_FC=False, CMFClassifier=False)
orig_model = get_model(args_pt, device)
orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
orig_model.eval()
print('\n-- Original model accuracy (3/7 split) --')
orig_ra, orig_fa, _ = test(orig_model, device, test_loader, FORGET_CLASSES,
                            CLASS_LABEL_NAMES, NUM_CLASSES,
                            plot_cm=False, job_name='original', set_name='Test')
print(f'  retain={orig_ra:.4f}  forget={orig_fa:.4f}')
print('\nData loaded.')

## D. Run Group B Ablation

Runs Random-label and SalUn each under `mean_source='train'` and `mean_source='retain'`.

In [ ]:
sys.path.insert(0, REPO_DIR)
from experiments.cmf_loader_ablation.ablation_model import AblationModelModule
from experiments.cmf_loader_ablation.ablation_unlearn import (
    ablation_random_label_CMF,
    ablation_salun_CMF,
)

ABLATION_FN = {
    'random_label_CMF_RemoveFC': ablation_random_label_CMF,
    'salun_CMF_RemoveFC':        ablation_salun_CMF,
}
print('Ablation components loaded.')

In [ ]:
import time

RESULTS_B = []
MODEL_STORE = {}

def _load_cmf_model(args):
    model = AblationModelModule(args).to(device)
    sd = torch.load(CKPT_CMF_FT, map_location=device)
    if isinstance(sd, dict) and 'state_dict' in sd:
        sd = sd['state_dict']
    model.load_state_dict(sd, strict=False)
    return model

def run_one(method, mean_source):
    print(f'\n{"="*65}')
    print(f'  METHOD      : {method}')
    print(f'  mean_source : {mean_source}')
    print(f'  forget      : {FORGET_CLASSES}  (3/7 split)')
    print(f'{"="*65}')

    lr     = get_lr(method)
    epochs = get_epochs(method)

    args = make_args(
        unlearn_method=method, epochs_or_steps=epochs, lr=lr,
        batch_size=UNLEARN_BS, num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        num_retain_samples=NUM_RETAIN, num_forget_samples=NUM_FORGET,
        unlearn_class=list(FORGET_CLASSES),
        remove_FC=True, CMFClassifier=True, lp_every=0, ncc_every=0,
    )

    model     = _load_cmf_model(args)
    optimizer = optim.SGD(model.parameters(), lr=lr,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)
    t0 = time.time()
    model = ABLATION_FN[method](
        args=args, model=model, device=device,
        retain_loader=retain_loader, forget_loader=forget_loader,
        train_loader=train_loader,
        retain_loader_full=retain_full_loader,
        test_loader=test_loader, optimizer=optimizer, epochs=epochs,
        test_forget_loader=test_forget_loader,
        mean_source=mean_source,
    )
    t_train = time.time() - t0

    model.eval()
    model.recompute_cmf(loader_full=train_loader, loader_retain=retain_full_loader,
                        device=device, mean_source=mean_source)

    ra, fa, _ = test(model, device, test_loader, FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
                     plot_cm=False, job_name=method, set_name='Final Test')

    import evaluation
    def _get_model_fn(a): return AblationModelModule(a).to(device)
    lp_outs = evaluation.run_linear_probe_on_fresh_clone(
        args=args, get_model_fn=_get_model_fn, device_probe=device,
        src_model=model, train_loader=train_loader, test_loader=test_loader,
        num_classes=NUM_CLASSES, bs_probe=128,
    )

    from evaluation.nc_cmf import ncc_mismatch
    ncc_r = ncc_mismatch(args=args, model=model, train_loader=train_loader,
                         eval_loader=test_loader,        device=device, pool_mode='avg')
    ncc_f = ncc_mismatch(args=args, model=model, train_loader=train_loader,
                         eval_loader=test_forget_loader, device=device, pool_mode='avg')

    history  = getattr(model, 'history_log', {})
    mu_drift = history.get('mu_drift', [])

    row = dict(
        method=method, mean_source=mean_source,
        output_retain_acc=float(ra), output_forget_acc=float(fa),
        probe_retain_acc=float(lp_outs['acc_test_retain']),
        probe_forget_acc=float(lp_outs['acc_test_forget']),
        ncc_retain_acc=float(ncc_r['ncc_acc']),
        ncc_forget_acc=float(ncc_f['ncc_acc']),
        mu_drift_avg=float(np.mean(mu_drift))  if mu_drift else float('nan'),
        mu_drift_max=float(np.max(mu_drift))   if mu_drift else float('nan'),
        mu_drift_per_epoch=mu_drift,
        history=history, wall_time_s=round(t_train, 1),
    )

    print(f'  Output : retain={ra:.4f}  forget={fa:.4f}')
    print(f'  LP     : retain={lp_outs["acc_test_retain"]:.4f}  forget={lp_outs["acc_test_forget"]:.4f}')
    print(f'  NCC    : retain={ncc_r["ncc_acc"]:.4f}  forget={ncc_f["ncc_acc"]:.4f}')
    print(f'  mu_drift: avg={row["mu_drift_avg"]:.4f}  max={row["mu_drift_max"]:.4f}')

    ckpt_out = (f'{WORK_ROOT}/checkpoints/{method}_{mean_source}/'
                f'{DATASET}_{ARCH}_{_MODE_TAG}/{_FORGET_STR}.pt')
    os.makedirs(os.path.dirname(ckpt_out), exist_ok=True)
    torch.save(model.state_dict(), ckpt_out)

    return row, model

for method in RUN_METHODS:
    for mean_source in MEAN_SOURCES:
        try:
            row, trained_model = run_one(method, mean_source)
            RESULTS_B.append(row)
            MODEL_STORE[(method, mean_source)] = trained_model
        except Exception as e:
            import traceback; traceback.print_exc()
            RESULTS_B.append(dict(
                method=method, mean_source=mean_source,
                output_retain_acc=float('nan'), output_forget_acc=float('nan'),
                probe_retain_acc=float('nan'),  probe_forget_acc=float('nan'),
                ncc_retain_acc=float('nan'),    ncc_forget_acc=float('nan'),
                mu_drift_avg=float('nan'),      mu_drift_max=float('nan'),
                mu_drift_per_epoch=[], history={}, wall_time_s=0,
            ))

print('\nGroup B ablation runs complete.')

## E. Results Table

In [ ]:
rt_csv = os.path.join(CKPT_ROOT, 'retrain_results.csv')
rt_row = pd.read_csv(rt_csv).iloc[0].to_dict() if os.path.exists(rt_csv) else \
         {'retain_acc': float('nan'), 'forget_acc': float('nan')}

df = pd.DataFrame(RESULTS_B)

baselines = pd.DataFrame([
    dict(method='Original', mean_source='-',
         output_retain_acc=orig_ra, output_forget_acc=orig_fa,
         probe_retain_acc=float('nan'), probe_forget_acc=float('nan'),
         ncc_retain_acc=float('nan'),  ncc_forget_acc=float('nan'),
         mu_drift_avg=float('nan'),    mu_drift_max=float('nan')),
    dict(method='Retrain', mean_source='-',
         output_retain_acc=rt_row['retain_acc'], output_forget_acc=rt_row['forget_acc'],
         probe_retain_acc=float('nan'), probe_forget_acc=float('nan'),
         ncc_retain_acc=float('nan'),  ncc_forget_acc=float('nan'),
         mu_drift_avg=float('nan'),    mu_drift_max=float('nan')),
])
display_df = pd.concat([baselines, df], ignore_index=True)

METRIC_COLS = ['output_retain_acc','output_forget_acc',
               'probe_retain_acc','probe_forget_acc',
               'ncc_retain_acc','ncc_forget_acc',
               'mu_drift_avg','mu_drift_max']

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 160)
print(f'\n=== Group B Results — {DATASET}/{ARCH} | Forget={FORGET_CLASSES} (3/7) ===')
print(display_df[['method','mean_source'] + METRIC_COLS].to_string(index=False))

print('\n=== Delta (retain - train) ===')
for method in RUN_METHODS:
    r_t = df[(df.method==method)&(df.mean_source=='train')]
    r_r = df[(df.method==method)&(df.mean_source=='retain')]
    if r_t.empty or r_r.empty: continue
    short = method.replace('_CMF_RemoveFC','').replace('_',' ')
    for col in ['output_retain_acc','output_forget_acc','mu_drift_avg']:
        d = float(r_r[col].iloc[0]) - float(r_t[col].iloc[0])
        print(f'  {short:30s}  d_{col:25s} = {d:+.4f}')

csv_path = f'{WORK_ROOT}/ablation_results_grpB_{DATASET}_{ARCH}.csv'
display_df.drop(columns=['mu_drift_per_epoch','history'], errors='ignore').to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

## F. μ Drift Plot

In [ ]:
colors  = {'train': 'tomato', 'retain': 'steelblue'}
markers = {'train': 'o',      'retain': 's'}

fig, axes = plt.subplots(1, len(RUN_METHODS), figsize=(5 * len(RUN_METHODS), 3.5))
if len(RUN_METHODS) == 1: axes = [axes]

for ax, method in zip(axes, RUN_METHODS):
    plotted = False
    for mean_source in MEAN_SOURCES:
        rows = [r for r in RESULTS_B if r['method']==method and r['mean_source']==mean_source]
        if not rows: continue
        drifts = rows[0].get('mu_drift_per_epoch', [])
        if not drifts: continue
        ax.plot(range(1, len(drifts)+1), drifts,
                color=colors[mean_source], marker=markers[mean_source],
                markersize=5, linewidth=1.5, label=f'mu_source={mean_source}')
        plotted = True
    short = method.replace('_CMF_RemoveFC','').replace('_',' ')
    ax.set_title(short, fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('||mu_e - mu_{e-1}||_2', fontsize=8)
    if plotted: ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Group B mu drift | {DATASET}/{ARCH} | forget={FORGET_CLASSES} (3/7)', fontsize=10)
plt.tight_layout()
fig_path = f'{WORK_ROOT}/ablation_drift_grpB_{DATASET}_{ARCH}.png'
plt.savefig(fig_path, dpi=130); plt.show()
print('Saved:', fig_path)

## G. Bar Chart

In [ ]:
methods_p = [m for m in RUN_METHODS if m in df['method'].values]
n, x, w = len(methods_p), np.arange(len(methods_p)), 0.18

fig, axes = plt.subplots(1, 2, figsize=(max(7, 3*n), 4.5))

for ax_i, (mc_r, mc_f, title) in enumerate([
    ('output_retain_acc','output_forget_acc','Output-Level'),
    ('probe_retain_acc', 'probe_forget_acc', 'Linear-Probe'),
]):
    ax = axes[ax_i]
    ax.axhline(orig_ra, color='grey', ls='--', lw=1, alpha=0.6, label='Orig retain')
    ax.axhline(orig_fa, color='grey', ls=':',  lw=1, alpha=0.6, label='Orig forget')
    ax.axhline(rt_row['retain_acc'], color='k', ls='--', lw=1, alpha=0.6, label='Retrain retain')

    for i, method in enumerate(methods_p):
        for src, off, rc, fc in [('train', -1.5*w, 'steelblue', 'lightcoral'),
                                  ('retain', +0.5*w, 'royalblue', 'tomato')]:
            row_m = df[(df.method==method)&(df.mean_source==src)]
            rv = float(row_m[mc_r].iloc[0]) if not row_m.empty else float('nan')
            fv = float(row_m[mc_f].iloc[0]) if not row_m.empty else float('nan')
            for val, offset, color, lbl in [(rv, off,   rc, f'{src}/retain'),
                                             (fv, off+w, fc, f'{src}/forget')]:
                b = ax.bar(i+offset, val, w, color=color, alpha=0.82,
                           label=lbl if i==0 else '_nolegend_')
                if not np.isnan(val):
                    ax.text(b[0].get_x()+b[0].get_width()/2, val+0.005,
                            f'{val:.2f}', ha='center', va='bottom', fontsize=6)

    short_names = [m.replace('_CMF_RemoveFC','').replace('_',' ') for m in methods_p]
    ax.set_xticks(x); ax.set_xticklabels(short_names, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.12)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=6, ncol=2); ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Group B | mu-source: train vs retain | {DATASET}/{ARCH} | forget={FORGET_CLASSES}', fontsize=9)
plt.tight_layout()
fig_path = f'{WORK_ROOT}/ablation_bar_grpB_{DATASET}_{ARCH}.png'
plt.savefig(fig_path, dpi=130); plt.show()
print('Saved:', fig_path)

## H. t-SNE

In [ ]:
from sklearn.manifold import TSNE

def collect_features(model, loader, max_pts=2000):
    model.eval(); feats, labs, seen = [], [], 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            z = model.extract_features(x).cpu() if hasattr(model,'extract_features') else model(x).cpu()
            feats.append(z.numpy()); labs.append(y.numpy())
            seen += x.size(0)
            if seen >= max_pts: break
    return np.concatenate(feats)[:max_pts], np.concatenate(labs)[:max_pts]

def plot_tsne(model, loader, title, forget_cls, max_pts=1500, ax=None):
    feats, labs = collect_features(model, loader, max_pts)
    emb = TSNE(n_components=2, perplexity=30, random_state=0, max_iter=500).fit_transform(feats)
    if ax is None: _, ax = plt.subplots(figsize=(5,4))
    cmap = plt.cm.get_cmap('tab10', NUM_CLASSES)
    for c in sorted(set(labs)):
        mask = labs==c
        ax.scatter(emb[mask,0], emb[mask,1], s=7,
                   alpha=0.7 if c in forget_cls else 0.4,
                   color='red' if c in forget_cls else cmap(c),
                   zorder=5 if c in forget_cls else 1,
                   label=CLASS_LABEL_NAMES[c] if c in forget_cls else '_nolegend_')
    ax.set_title(title, fontsize=9); ax.axis('off'); return ax

tsne_method = next((m for m in RUN_METHODS
                    if (m,'train') in MODEL_STORE and (m,'retain') in MODEL_STORE), None)

if tsne_method is None:
    print('No complete (train+retain) pair available for t-SNE.')
else:
    vis_loader = torch.utils.data.DataLoader(dataset_test, batch_size=256, shuffle=True, num_workers=2)
    fig, axes  = plt.subplots(1, 3, figsize=(15, 4.5))
    plot_tsne(orig_model, vis_loader, 'Original', FORGET_CLASSES, ax=axes[0])
    axes[0].legend(loc='lower right', fontsize=7, markerscale=2, title='Forget class')
    short = tsne_method.replace('_CMF_RemoveFC','').replace('_',' ')
    for ax, src in zip(axes[1:], ['train','retain']):
        plot_tsne(MODEL_STORE[(tsne_method,src)], vis_loader,
                  f'{short}\nmu_source={src}', FORGET_CLASSES, ax=ax)
        ax.legend(loc='lower right', fontsize=7, markerscale=2, title='Forget class')
    plt.suptitle(f't-SNE Group B | {DATASET}/{ARCH} | forget={FORGET_CLASSES} (red=forgotten)', fontsize=10)
    plt.tight_layout()
    fig_path = f'{WORK_ROOT}/ablation_tsne_grpB_{DATASET}_{ARCH}.png'
    plt.savefig(fig_path, dpi=130); plt.show()
    print('Saved:', fig_path)

## I. Combined Summary — All Methods (Group A + Group B)

If `GRPA_RESULTS_CSV` is set (Group A output attached as a dataset), this section  
merges both groups into one final table and written summary — matching the paper's  
combined Table 3 format.

In [ ]:
# Load Group A CSV if available
if GRPA_RESULTS_CSV and os.path.exists(GRPA_RESULTS_CSV):
    df_a = pd.read_csv(GRPA_RESULTS_CSV)
    # Drop baseline rows (method in ['Original','Retrain']) to avoid duplication
    df_a = df_a[~df_a['method'].isin(['Original', 'Retrain'])]
    print(f'Loaded Group A results: {len(df_a)} rows from {GRPA_RESULTS_CSV}')
else:
    df_a = pd.DataFrame()
    print('Group A CSV not found — showing Group B only.')

df_b_clean = df.copy()
df_combined = pd.concat([df_a, df_b_clean], ignore_index=True)

# Add baselines
all_display = pd.concat([baselines, df_combined], ignore_index=True)

print(f'\n=== FULL Ablation Results — {DATASET}/{ARCH} | Forget={FORGET_CLASSES} (3/7 split) ===')
print(all_display[['method','mean_source'] + METRIC_COLS].to_string(index=False))

# Per-method delta table
ALL_METHODS = list(df_combined['method'].unique())
ORDER = ['scrub_CMF_RemoveFC','grad_ascent_descent_CMF_RemoveFC',
         'random_label_CMF_RemoveFC','salun_CMF_RemoveFC']
ALL_METHODS = [m for m in ORDER if m in ALL_METHODS] + \
              [m for m in ALL_METHODS if m not in ORDER]

print('\n=== Combined Delta (retain - train) per method ===')
THR = 0.005
for method in ALL_METHODS:
    r_t = df_combined[(df_combined.method==method)&(df_combined.mean_source=='train')]
    r_r = df_combined[(df_combined.method==method)&(df_combined.mean_source=='retain')]
    if r_t.empty or r_r.empty:
        print(f'  {method}: incomplete data')
        continue
    short = method.replace('_CMF_RemoveFC','').replace('_',' ')
    d_ra  = float(r_r['output_retain_acc'].iloc[0]) - float(r_t['output_retain_acc'].iloc[0])
    d_fa  = float(r_r['output_forget_acc'].iloc[0]) - float(r_t['output_forget_acc'].iloc[0])
    d_mu  = float(r_r['mu_drift_avg'].iloc[0])       - float(r_t['mu_drift_avg'].iloc[0])
    ra_verdict  = '[+]' if d_ra >  THR else ('[-]' if d_ra < -THR else '[~]')
    fa_verdict  = '[!]' if abs(d_fa) > THR else '[=]'
    mu_verdict  = '[+]' if d_mu < -1e-4 else '[~]'
    print(f'  {short:35s}  retain={d_ra:+.4f} {ra_verdict}  '
          f'forget={d_fa:+.4f} {fa_verdict}  mu_drift={d_mu:+.4f} {mu_verdict}')

# Save combined CSV
combined_csv = f'{WORK_ROOT}/ablation_results_combined_{DATASET}_{ARCH}.csv'
all_display.drop(columns=['mu_drift_per_epoch','history'], errors='ignore').to_csv(combined_csv, index=False)
print(f'\nCombined results saved: {combined_csv}')

print('\n' + '='*65)
print('ABLATION COMPLETE')
print('='*65)
print('Legend: [+] improved  [-] hurt  [~] negligible  [=] unchanged  [!] unexpected change')